<a href="https://colab.research.google.com/github/Emakporpaul/FlyRank-ML-Internship---Starting-Repository/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emakporpaul/FlyRank-ML-Internship---Starting-Repository/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane 2 - Refresh / Content Opportunity Scoring:

 Over Weeks 1-2 I ran the starter pipeline and saw a learned model clearly outperform a hand-written rule at the same task: the baseline rule scored 0.240 Precision@50, while the random forest scored 0.740 - roughly 37 of the top 50 flagged pages were actually declining, versus about 12 for the rule (outputs/model_results.json). That's exactly the Lane 2 question - which pages should be reviewed first - and the gap between rule and model is large enough to be worth seven more weeks of work, not a rounding error. I also want to build past the starter's proxy label (trend_direction == "down", a current-window bucket) toward a real future-window label, as the guide recommends for a stronger capstone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "still not at repo root"

Working dir: /content/flyrank-ml-internship-starter


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_all.py'], returncode=0)

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))
print("Baseline Precision@50:", res["baseline"]["baseline_precision_at_50"])
print("Random forest Precision@50:", res["models"]["random_forest"]["precision_at_50"])
print("Validation design used:", res["split_strategy"])


Baseline Precision@50: 0.24
Random forest Precision@50: 0.74
Validation design used: client_holdout


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Decision: out of all pages with real traffic, which ones should a content team
review first this cycle for refresh, expansion, or pruning?

Unit of analysis: one page (content_id in the starter data; content_hash_id in
the warehouse release).

Who acts on it: a content editor or reviewer, working down a ranked queue with
reason codes attached to each page.

Cost of a wrong call:
- False positive (page ranked high, wasn't actually a priority): wastes limited
  editor hours on a page that didn't need review.
- False negative (a genuinely declining page ranked low or missed): the page's
  traffic loss goes unaddressed for another review cycle, which is more costly
  since editor time is the scarce resource, not page count.

Because review capacity is limited, this points to optimizing for Precision@K
(K = however many pages the team can actually act on per cycle) rather than
raw accuracy across all pages.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter slice that make this lane worth pursuing:
1. A learned model already beats a hand-written rule at the exact task this lane cares about (ranking pages to review first).
2. Search volume alone is a poor proxy for actual traffic — meaning simple demand-based rules leave real signal on the table.
3. Click-Through Rate (CTR) varies sharply by position, so any scoring approach needs to account for position rather than treating CTR as directly comparable across pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
import pandas as pd

# 1. Rule vs. model gap on the exact task this lane targets
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"1) Baseline Precision@50: {base:.3f}   Random forest Precision@50: {rf:.3f}")
print(f"   -> model gets ~{round(rf*50)} of top 50 right vs ~{round(base*50)} for the rule")

# 2. Search volume vs. actual traffic
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"2) search_volume vs impressions_90d correlation: {corr:.3f} (near zero)")

# 3. CTR range across position tiers
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(f"3) CTR by position tier ranges from {ctr_by_pos.iloc[0]:.3f} (best) to {ctr_by_pos.iloc[-1]:.3f} (worst)")


1) Baseline Precision@50: 0.240   Random forest Precision@50: 0.740
   -> model gets ~37 of top 50 right vs ~12 for the rule
2) search_volume vs impressions_90d correlation: 0.001 (near zero)
3) CTR by position tier ranges from 0.355 (best) to 0.055 (worst)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim:
- Observed, in-sample patterns from the anonymized starter slice (e.g. the
  gap between a hand rule and a learned model, CTR by position, search
  volume vs. traffic).
- A directional ranking of pages worth reviewing first, backed by
  reason codes a human can inspect and disagree with.
- A decision-support tool - it surfaces candidates for a reviewer's
  limited time, it does not make the call itself.
- Results validated with client-holdout splits (a client's pages never
  appear in both train and test), which is stronger than a plain random
  split but still not a live experiment.

What I will never claim:
- That a refresh CAUSES a page to recover - that requires an actual
  experiment or causal design, which this data alone cannot provide.
- That any result "predicts Google's algorithm" or reveals a ranking
  factor — this data shows correlations in observed search behavior,
  not how Google's systems work internally.
- That the starter's proxy label (trend_direction == "down", a
  current-window bucket) is the true target - it's a known-weak
  stand-in I plan to replace with a future-window label.
- That in-sample numbers generalize beyond this slice until re-validated
  on a proper train/test or client-holdout split.
- That a declining page is guaranteed to be worth fixing, or that a
  high-scoring page is guaranteed to improve if reviewed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
res = json.load(open("outputs/model_results.json"))
print("Validation design used for the numbers cited above:", res["split_strategy"])


Validation design used for the numbers cited above: client_holdout


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.